In [105]:
import argparse
import requests
import re
import html
import time
import sys
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from multiprocessing import Pool, cpu_count


def get_usi_from_mgf(mgf):

    """Extract USI from MGF file content."""
    usi = None
    for line in mgf.split('\n'):
        line = line.strip()
        if line.startswith('PROVENANCE_USI='):
            usi = line.split('=', 1)[1]
            break
    return usi


def compute_theoretical_mass_from_usi(usi):
    try:
        # extract peptide part after the 'scan:<number>:' segment of the USI
        m = re.search(r'scan:\d+:(.+)$', usi)
        if m:
            peptide_part = m.group(1)
        else:
            # fallback: take last colon-separated field (legacy USIs)
            peptide_part = usi.rsplit(':', 1)[-1]
        peptide = peptide_part.split('/')[0]
        # print(peptide)
        aa_mono = {
            'A':71.037114,'R':156.101111,'N':114.042927,'D':115.026943,'C':103.009185,
            'E':129.042593,'Q':128.058578,'G':57.021464,'H':137.058912,'I':113.084064,
            'L':113.084064,'K':128.094963,'M':131.040485,'F':147.068414,'P':97.052764,
            'S':87.032028,'T':101.047679,'W':186.079313,'Y':163.063329,'V':99.068414
        }
        # expanded set of common modification names -> monoisotopic mass shifts
        mod_masses = {
            # alkylation / cysteine
            'Carbamidomethyl': 57.021464,
            'Carboxymethyl': 58.005479,
            'Dicarbamidomethyl': 114.042927,
            'Pyro-carbamidomethyl': 39.994915,

            # oxidation / common variable mods
            'Oxidation': 15.994915,
            'Ox': 15.994915,

            # phosphorylation
            'Phospho': 79.966331,
            'Phosphorylation': 79.966331,

            # acetylation / N-term
            'Acetyl': 42.010565,
            'Acetylation': 42.010565,

            # deamidation
            'Deamidation': 0.984016,
            'Deamidated': 0.984016,
            'Deamidated:18O(1)': 2.988261,

            # ubiquitin remnant
            'GlyGly': 114.042927,
            'GlyGlyRemnant': 114.042927,
            'GG': 114.042927,
            'LRGG': 383.228103,

            # methylation series
            'Methyl': 14.015650,
            'Dimethyl': 28.031300,
            'Dimethyl:2H(4)': 32.056407,
            'Dimethyl:2H(4)13C(2)': 34.063117,
            'Dimethyl:2H(6)13C(2)': 36.075670,
            'Trimethyl': 42.046950,

            # small additions / others
            'Formyl': 27.994915,
            'Sulfation': 79.956815,
            'Pyro-glu': -17.026549,
            'PyroGlu': -17.026549,
            'Gln->pyro-Glu': -17.026549,
            'Glu->pyro-Glu': -18.010565,
            'Propionyl': 56.026215,
            'Propionamide': 71.037114,
            'Carbamyl': 43.005814,
            'Dethiomethyl': -48.003371,
            'Ammonia-loss': -17.026549,
            'Methylthio': 45.987721,
            'Cysteinyl': 119.004099,
            'Nethylmaleimide': 125.047679,
            'Thiazolidine': 87.998285,

            # common labeling reagents (mass added to peptide backbone)
            'TMT6plex': 229.162932,
            'TMT6plex114': 229.162932,
            'TMT10plex': 229.162932,
            'TMT11plex': 229.162932,
            'TMT16plex': 229.162932,
            'TMTpro': 304.207146,

            'iTRAQ4plex': 144.102063,
            'iTRAQ4plex114': 144.102063,
            'iTRAQ8plex': 304.205360,
            'iTRAQ8plex:13C(6)15N(2)': 304.205360,

            'DiLeu4plex117': 145.125595,

            # isotope labels
            'Label:13C(6)': 6.020129,
            'Label:13C(6)15N(1)': 7.017164,
            'Label:13C(6)15N(2)': 8.014199,
            'Label:13C(6)15N(4)': 10.008269,
            'Label:2H(4)': 4.025107,

            # ADP-Ribosyl
            'ADP-Ribosyl': 541.061110,

            # numeric modifications (use exact values from your data)
            '+141.11544': 141.11544,
            '+141.1154': 141.1154,
            '+1431.83104': 1431.83104,
            '+1541.85014': 1541.85014,
            '+1555.95614': 1555.95614,
            '+186.1127': 186.1127,
            '+186.1165': 186.1165,
            '+229.162931': 229.162931,
            '+271.1735': 271.1735,
            '+271.1736': 271.1736,
            '+454.18121': 454.18121,
            '+46.03274': 46.03274,
            '+46.0328': 46.0328,
            '+471.20776': 471.20776,
            '+64.10696': 64.10696,
            '+75.04729': 75.04729,
            '+85.05549': 85.05549,
            '0.0233': 0.0233,
            'Xlink:BuUrBu[85]': 85.0,
            # Label:13C(6) mods
            'Label:13C(6)': 6.020129,
            'Label:13C(6)15N(1)': 7.017164,
            'Label:13C(6)15N(2)': 8.014199,
            'Label:13C(6)15N(4)': 10.008269,

            # Deamidated with 18O
            'Deamidated:18O(1)': 2.988261,
        }

        # simpler approach: find all bracketed modifications and all residues, sum them
        total = 18.010564  # + H2O

        # extract bracketed modifications (e.g. C[Carbamidomethyl], [Acetyl]-...)
        bracket_mods = re.findall(r'\[([^\]]+)\]', peptide)
        mods_total = 0.0
        for mod in bracket_mods:
            mods_total += mod_masses.get(mod, 0.0)

        # remove bracketed parts to avoid double counting when searching for numeric mods / residues
        peptide_clean = re.sub(r'\[[^\]]+\]', '', peptide)

        # catch standalone numeric modifications outside brackets like +141.11544
        numeric_mods = re.findall(r'([+-]\d+\.\d+)', peptide_clean)
        for nm in numeric_mods:
            try:
                mods_total += float(nm)
            except ValueError:
                pass

        # find all residue single-letter codes in the cleaned peptide
        residues = re.findall(r'[A-Z]', peptide_clean)
        aa_total = sum(aa_mono.get(res, 0.0) for res in residues)

        total += aa_total + mods_total

        # maintain original behavior (adds one proton)
        return total + 1.007276
    except Exception:
        return None


def compare_mass_from_mgf(mgf_content):
    """Extract theoretical mass from MGF and compare with PEPMASS, return USI if different."""
    usi = get_usi_from_mgf(mgf_content)

    if not usi:
        return None
    
    theoretical_mass = compute_theoretical_mass_from_usi(usi)
    if theoretical_mass is None:
        return None
    
    # Extract PEPMASS from MGF
    pepmass = None
    charge = int(usi[-1])
    # Initialize charge before extracting from MGF
    for line in mgf_content.split('\n'):
        line = line.strip()
        if line.startswith('PEPMASS='):
            pepmass = float(line.split('=', 1)[1])
        # elif line.startswith('CHARGE='):
        #     charge_str = line.split('=', 1)[1]
        #     charge = int(charge_str.rstrip('+'))
    
    if pepmass is None or charge is None:
        return None
    
    # Convert theoretical monoisotopic mass to m/z
    # if charge == 0:
    #     print(usi)
    theoretical_mz = theoretical_mass / charge
    
    # Compare masses (using a tolerance, e.g., 0.1 Da)
    tolerance = 3
    if abs(theoretical_mz - pepmass) > tolerance:
        print(f"Discrepancy found for USI: {usi}", theoretical_mz, pepmass)
        return (usi,  theoretical_mz, pepmass)
    
    return None

In [106]:
usi_example = "mzspec:dummy:dummy:scan:1:PPR[Label:13C(6)15N(4)]SNR[Label:13C(6)15N(4)]FTAEEGDLGFTLR/3"
usi_example = 'mzspec:PXD002765:20150617_QE3_UPLC11_AKP_SA_COLLAB_ASSET_WP8_Asp14_Mix3_TiO2_HpH_02:scan:13862:ASGC[Carbamidomethyl]LAR[Label:13C(6)15N(4)]PGPPPSPGAAS[Phospho]DDDDDDVVGATPPAR[Label:13C(6)15N(4)]/3'
# usi_example = 'mzspec:PXD002765:20150617_QE3_UPLC11_AKP_SA_COLLAB_ASSET_WP8_Asp14_Mix3_TiO2_HpH_02:scan:13862:ASGC[Carbamidomethyl]LAR[Label:13C(6)15N(4)]PGPPPSPGAAS[Phospho]DDDDDDVVGATPPAR/3'
theoretical_mass = compute_theoretical_mass_from_usi(usi_example)
print(theoretical_mass)
#printed 1672.763548 but it should be 2183.0643, why?

3288.4377809999996


In [107]:
# Read the MGF file and split into individual spectra
with open('Normal_1_2_3.mgf', 'r') as f:
    content = f.read()

# Split by "BEGIN IONS" to get individual spectra
mgf_blocks = content.split('BEGIN IONS')
mgf_blocks = [block.strip() for block in mgf_blocks if block.strip()]

# Process each MGF block and collect USIs with mass differences
error_usis = []
for block in tqdm(mgf_blocks, desc="Processing spectra"):
    mgf_content = 'BEGIN IONS\n' + block
    result = compare_mass_from_mgf(mgf_content)
    if result is not None:
        usi, theoretical_mz, pepmass = result
        error_usis.append((theoretical_mz, pepmass, usi))

# Write error USIs to file
with open('errors_normal.txt', 'w') as f:
    for theoretical_mz, pepmass, usi in error_usis:
        f.write(f"{theoretical_mz}, {pepmass}, {usi}\n")

# print(f"Found {len(error_usis)} spectra with mass differences")

Processing spectra:  11%|█         | 2684/24559 [00:00<00:01, 12718.28it/s]

Discrepancy found for USI: mzspec:PXD007088:T05:scan:10816:[Acetyl]-MEPW[Thiazolidine]KPQHSFFFLLLLWLPDTTGEIVM[Oxidation]TQSPATLSLSPGER/5 969.2793846000002 955.0855
Discrepancy found for USI: mzspec:PXD007088:N02:scan:11318:[Acetyl]-MEPW[Thiazolidine]KPQHSFFFLLLLWLPDTTGEIVM[Oxidation]TQSPATLSLSPGER/5 969.2793846000002 954.901
Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:78536:ALGDFALFK/2 490.770197 570.60298
Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:226218:AFSQSSSLR/2 491.2476165 484.26974
Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:94918:TGQC[Carbamidomethyl]IC[Carbamidomethyl]KPNVEGR/3 506.2383863333334 711.35775
Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:385042:GAVVNLASVSSGAVR/2 693.3849759999999 973.50506
Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:122580:LYQVALTNR/2 538.80256 524.60455
Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:sc

Processing spectra:  21%|██        | 5139/24559 [00:00<00:01, 11204.93it/s]

Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:34533:FHGEPQKPPMN[Deamidated]GYHK/4 441.956788 744.37739
Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:278544:ESWPQYSQMYPGMR/2 879.8783445 725.89234
Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:109867:TLSPENYAAYK/2 628.307872 414.56619
Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:395317:TLSPENYAAYK/2 628.307872 827.93833
Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:328562:WLEISC[Carbamidomethyl]NLR/2 595.2993264999999 575.27773
Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:80823:YLMETWPFGELLC[Carbamidomethyl]K/2 893.4271389999999 794.93182
Discrepancy found for USI: mzspec:PXD021482:20200731_U118_40:scan:312483:LLQETSEQLVALKPWITR/3 708.4005223333334 811.90879


Processing spectra:  60%|██████    | 14831/24559 [00:01<00:00, 11368.72it/s]

...

Processing spectra:  77%|███████▋  | 18918/24559 [00:01<00:00, 12822.45it/s]

Discrepancy found for USI: mzspec:PXD006471:run1-07-Sample19:scan:3814:TPSPQLC[Carbamidomethyl]TSSPPR/2 713.84737 506.2535
Discrepancy found for USI: mzspec:PXD002255:ES_XP_MB_378:scan:19129:GGC[Carbamidomethyl]GSC[Carbamidomethyl]GGSKGGC[Carbamidomethyl]GSC[Carbamidomethyl]GC[Carbamidomethyl]SQC[Carbamidomethyl]SC[Carbamidomethyl]YKPC[Carbamidomethyl]C[Carbamidomethyl]C[Carbamidomethyl]SSGC[Carbamidomethyl]GSSC[Carbamidomethyl]C[Carbamidomethyl]QSSC[Carbamidomethyl]C[Carbamidomethyl]KP/4 1250.40922225 1403.46985
Discrepancy found for USI: mzspec:PXD002255:ES_XP_LMW_Proteominer_203:scan:14784:GGC[Carbamidomethyl]GSC[Carbamidomethyl]GGSKGGC[Carbamidomethyl]GSC[Carbamidomethyl]GC[Carbamidomethyl]SQC[Carbamidomethyl]SC[Carbamidomethyl]YKPC[Carbamidomethyl]C[Carbamidomethyl]C[Carbamidomethyl]SSGC[Carbamidomethyl]GSSC[Carbamidomethyl]C[Carbamidomethyl]QSSC[Carbamidomethyl]C[Carbamidomethyl]KP/5 1000.3273778 1123.16479
Discrepancy found for USI: mzspec:PXD001406:GM19114_MSB14197_27:scan:1217

Processing spectra:  95%|█████████▍| 23321/24559 [00:02<00:00, 13944.79it/s]

Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:79064:MMELTNVQQAVQALR/2 865.944019 898.46483
Discrepancy found for USI: mzspec:PXD021482:20200731_U118_40:scan:73793:MMELTNVQQAVQALR/2 865.944019 794.88803
Discrepancy found for USI: mzspec:PXD001559:20130926_VelosPro_RacMac_RM263_2_7:scan:967:NAER[Label:13C(6)15N(4)]EQES[Phospho]EEEM[Oxidation]/2 793.2775780000001 788.78
Discrepancy found for USI: mzspec:PXD006471:run1-19-Sample045:scan:23014:PSGALGNFSLEHGGLAGPEQYR/3 752.3665403333333 780.4047
Discrepancy found for USI: mzspec:PXD006471:run1-19-Sample045:scan:16380:EIPLC[Carbamidomethyl]AGC[Carbamidomethyl]DQHILDR/3 598.947272 731.3852
Discrepancy found for USI: mzspec:PXD021482:20200725_cell_40frac:scan:206987:EPGFVPGWDSFFEK/2 820.8791925 500.3133
Discrepancy found for USI: mzspec:PXD005445:QEp15-2721_HIP:scan:30823:QILHDTLSEAC[Carbamidomethyl]LRISEDERLR/4 638.5760525 642.0877


Processing spectra: 100%|██████████| 24559/24559 [00:02<00:00, 11443.08it/s]


In [108]:
# Read the MGF file and split into individual spectra
with open('Dimethyl_1_2_3.mgf', 'r') as f:
    content = f.read()

# Split by "BEGIN IONS" to get individual spectra
mgf_blocks = content.split('BEGIN IONS')
mgf_blocks = [block.strip() for block in mgf_blocks if block.strip()]

# Process each MGF block and collect USIs with mass differences
error_usis = []
for block in tqdm(mgf_blocks, desc="Processing spectra"):
    mgf_content = 'BEGIN IONS\n' + block
    result = compare_mass_from_mgf(mgf_content)
    if result is not None:
        usi, theoretical_mz, pepmass = result
        error_usis.append((theoretical_mz, pepmass, usi))

# Write error USIs to file
with open('errors_Dimethyl.txt', 'w') as f:
    for theoretical_mz, pepmass, usi in error_usis:
        f.write(f"{theoretical_mz}, {pepmass}, {usi}\n")

# Read the MGF file and split into individual spectra
with open('iTRAQ4_1_2_3.mgf', 'r') as f:
    content = f.read()

# Split by "BEGIN IONS" to get individual spectra
mgf_blocks = content.split('BEGIN IONS')
mgf_blocks = [block.strip() for block in mgf_blocks if block.strip()]

# Process each MGF block and collect USIs with mass differences
error_usis = []
for block in tqdm(mgf_blocks, desc="Processing spectra"):
    mgf_content = 'BEGIN IONS\n' + block
    result = compare_mass_from_mgf(mgf_content)
    if result is not None:
        usi, theoretical_mz, pepmass = result
        error_usis.append((theoretical_mz, pepmass, usi))

# Write error USIs to file
with open('errors_iTRAQ4.txt', 'w') as f:
    for theoretical_mz, pepmass, usi in error_usis:
        f.write(f"{theoretical_mz}, {pepmass}, {usi}\n")
#########################################
with open('iTRAQ8plex_1_2_3.mgf', 'r') as f:
    content = f.read()

# Split by "BEGIN IONS" to get individual spectra
mgf_blocks = content.split('BEGIN IONS')
mgf_blocks = [block.strip() for block in mgf_blocks if block.strip()]

# Process each MGF block and collect USIs with mass differences
error_usis = []
for block in tqdm(mgf_blocks, desc="Processing spectra"):
    mgf_content = 'BEGIN IONS\n' + block
    result = compare_mass_from_mgf(mgf_content)
    if result is not None:
        usi, theoretical_mz, pepmass = result
        error_usis.append((theoretical_mz, pepmass, usi))

# Write error USIs to file
with open('errors_iTRAQ8plex.txt', 'w') as f:
    for theoretical_mz, pepmass, usi in error_usis:
        f.write(f"{theoretical_mz}, {pepmass}, {usi}\n")

###################################
with open('TMT6plex_1_2_3.mgf', 'r') as f:
    content = f.read()

# Split by "BEGIN IONS" to get individual spectra
mgf_blocks = content.split('BEGIN IONS')
mgf_blocks = [block.strip() for block in mgf_blocks if block.strip()]

# Process each MGF block and collect USIs with mass differences
error_usis = []
for block in tqdm(mgf_blocks, desc="Processing spectra"):
    mgf_content = 'BEGIN IONS\n' + block
    result = compare_mass_from_mgf(mgf_content)
    if result is not None:
        usi, theoretical_mz, pepmass = result
        error_usis.append((theoretical_mz, pepmass, usi))

# Write error USIs to file
with open('errors_TMT6plex.txt', 'w') as f:
    for theoretical_mz, pepmass, usi in error_usis:
        f.write(f"{theoretical_mz}, {pepmass}, {usi}\n")
###################################
with open('TMTpro_1_2_3.mgf', 'r') as f:
    content = f.read()

# Split by "BEGIN IONS" to get individual spectra
mgf_blocks = content.split('BEGIN IONS')
mgf_blocks = [block.strip() for block in mgf_blocks if block.strip()]

# Process each MGF block and collect USIs with mass differences
error_usis = []
for block in tqdm(mgf_blocks, desc="Processing spectra"):
    mgf_content = 'BEGIN IONS\n' + block
    result = compare_mass_from_mgf(mgf_content)
    if result is not None:
        usi, theoretical_mz, pepmass = result
        error_usis.append((theoretical_mz, pepmass, usi))

# Write error USIs to file
with open('errors_TMTpro.txt', 'w') as f:
    for theoretical_mz, pepmass, usi in error_usis:
        f.write(f"{theoretical_mz}, {pepmass}, {usi}\n")


Processing spectra: 100%|██████████| 416/416 [00:00<00:00, 9327.60it/s]


Discrepancy found for USI: mzspec:PXD005177:120307_mHCT116_iTRAQ_09:scan:15406:[iTRAQ4plex114]-YMSRPVMSGPGLYDPTTIMNADILAY/3 1006.8229336666667 554.8256
Discrepancy found for USI: mzspec:PXD005177:120307_mHCT116_iTRAQ_02:scan:8579:[iTRAQ4plex114]-LEFQQQLGEAPSDASP/2 930.4547920000002 390.7216
Discrepancy found for USI: mzspec:PXD005177:120307_mHCT116_iTRAQ_02:scan:8587:[iTRAQ4plex114]-LEFQQQLGEAPSDASP/3 620.3031946666669 644.8462


Processing spectra: 100%|██████████| 3717/3717 [00:00<00:00, 11454.72it/s]


Discrepancy found for USI: mzspec:PXD007985:w036:scan:57736:[TMT6plex]-LETVPPLN[Deamidated]ELTEVPGEDK[TMT6plex]/2 1219.6635295 1500.1643


Processing spectra: 100%|██████████| 125/125 [00:00<00:00, 10855.95it/s]


BEGIN IONS
PEPMASS=562.33215
CHARGE=2
CHARGE_ID=2
MSLEVEL=2
COLLISION_ENERGY=0.0
FILENAME=
SEQ=+144.102063LLIYGGSTR
PROTEIN=
SCANS=1
SCAN=1
PROVENANCE_FILENAME=MSV000083107/peak/CPTAC_OvC_JB5429_iTRAQ_18_4Apr12_Cougar_12-03-22.mzML
PROVENANCE_SCAN=8441
PROVENANCE_USI=mzspec:PXD015899:CPTAC_OvC_JB5429_iTRAQ_18_4Apr12_Cougar_12-03-22:scan:8441:[iTRAQ4plex]-LLIYGGSTR/2
109.182129 5929.545410
110.071014 94182.390625
112.086472 10778.242188
113.070259 5820.926270
114.110420 224523.078125
115.086014 9933.619141
115.107567 147799.812500
116.070915 9409.044922
116.110825 472852.812500
117.114105 369053.343750
118.116966 10135.944336
120.080894 6206.313965
130.097229 5952.444824
133.042862 17103.792969
136.075439 63213.828125
143.109009 5263.282227
144.076645 12987.262695
145.108398 239428.640625
158.092041 62090.238281
161.092514 6021.440918
171.076553 16309.699219
175.118820 132363.218750
189.087540 9558.553711
195.086975 13855.679688
199.180389 60980.601562
202.082703 5549.318848
216.133316 6679.893555
219.360550 5137.445312
221.091354 17235.539062
227.175186 74394.476562
228.096970 10986.163086
228.178085 7385.122070
230.198074 86182.437500
235.118546 77923.398438
236.123505 9829.925781
239.741501 5186.695801
241.900177 5931.526855
249.159119 18523.798828
257.210083 6942.374512
258.193298 166470.218750
259.139526 50034.164062
259.196930 15892.166992
263.113708 70611.687500
264.116821 11122.745117
267.108643 6059.219238
276.168213 18793.238281
277.154114 12129.884766
278.111877 26778.859375
281.124146 7540.839844
285.119049 27301.341797
288.565186 4919.575684
291.214600 42189.542969
292.105499 8465.735352
301.200287 8597.721680
303.131592 11051.037109
310.117310 12763.040039
310.149536 8356.474609
318.154907 15089.695312
320.136353 14681.993164
328.162048 13214.366211
330.096466 4699.982910
334.173279 8928.315430
340.259216 13247.014648
343.283997 15987.450195
346.144592 8395.097656
346.174927 13112.525391
347.134125 15130.138672
359.206207 8874.750977
362.250732 9409.459961
363.200409 16210.381836
364.160706 17491.253906
365.143860 9918.435547
370.296661 12846.367188
371.276489 112987.195312
372.284363 15080.972656
391.195312 9311.309570
395.275055 5746.233398
402.045502 4928.267090
403.193573 21313.001953
407.159515 10003.834961
420.220856 8167.446777
425.178162 5804.172363
430.170044 8034.007324
430.296814 6497.419434
435.818878 5950.004395
442.204163 15223.819336
447.261810 6804.700684
448.178772 11994.923828
450.237579 9890.511719
452.663147 14690.655273
453.162109 6445.759277
456.362518 22547.970703
460.214020 42633.625000
461.218933 11511.370117
475.188293 12777.521484
475.344360 6703.023438
476.196228 7108.961426
477.240082 85769.953125
478.246521 17446.189453
484.362854 50336.785156
493.202972 14432.681641
562.320740 6442.905273
563.334778 18143.369141
563.408691 18742.724609
580.270996 9763.269531
587.246948 5829.050781
605.265686 15536.199219
606.251831 11119.519531
622.299072 8296.500000
623.278076 54580.199219
624.277527 16153.020508
635.222595 9672.064453
640.304749 160485.250000
641.307251 58474.757812
642.305847 8944.353516
650.301880 5129.616699
699.342773 21038.701172
700.354980 8302.072266
718.363586 6034.016602
723.316956 6012.760742
736.378235 12696.331055
753.389465 119191.718750
754.393372 53747.226562
795.256409 7773.374512
848.459351 8432.937500
849.451721 8457.331055
866.472534 161074.656250
867.474670 72385.414062
868.476379 16992.074219
962.541565 12222.014648
964.196960 5242.608887
978.527527 6646.722656
979.557434 183653.375000
980.559326 115244.554688
981.566406 29982.085938
END IONS

BEGIN IONS
PEPMASS=869.95246
CHARGE=4
CHARGE_ID=4
MSLEVEL=2
COLLISION_ENERGY=0.0
FILENAME=
SEQ=+144.105918ESGPALVK+144.105918PTQTLTLTC+57.021464TFSGFSLSTTGMR
PROTEIN=
SCANS=2
SCAN=2